# Employee Event Validation

This notebook validates employee-event history for the complete synthetic workforce.

The event table contains:

- Hires
- Promotions
- Transfers
- Manager changes
- Leave starts and returns
- Terminations

The main rules are:

- Event IDs are complete and unique.
- Employees must exist.
- Events must occur during employment.
- Every employee has exactly one hire event.
- Terminated employees have exactly one termination event.
- Active employees do not have termination events.
- Promotion events agree with compensation history.
- Transfers use valid location IDs.
- Manager changes use valid active managers.
- Leave events contain a start and return.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

locations = pd.read_csv(
    RAW_DATA_DIR / "locations.csv"
)

compensation_history = pd.read_csv(
    RAW_DATA_DIR / "compensation_history.csv",
    parse_dates=["effective_date"],
)

employee_events = pd.read_csv(
    RAW_DATA_DIR / "employee_events.csv",
    parse_dates=["event_date"],
)

employees["manager_id"] = (
    employees["manager_id"]
    .astype("Int64")
)

print("Employees:", employees.shape)
print(
    "Compensation history:",
    compensation_history.shape,
)
print(
    "Employee events:",
    employee_events.shape,
)

Employees: (10000, 15)
Compensation history: (30997, 7)
Employee events: (16008, 7)


## 1. Initial inspection

In [2]:
employee_events.head(10)

,event_id,employee_id,event_date,event_type,old_value,new_value,notes
0,1100001,100001,2022-10-17,Hire,NaN,Active,"Hired into department_id=1, location_id=1, job..."
1,1100002,100002,2022-04-03,Hire,NaN,Active,"Hired into department_id=1, location_id=1, job..."
2,1100003,100002,2024-04-15,Transfer,3,1,Internal location transfer; old_value and new_...
3,1100004,100002,2025-01-22,Leave,Active,Leave,Medical Leave; planned duration 90 days.
4,1100005,100002,2025-04-22,Leave,Leave,Active,Medical Leave; planned duration 90 days.
5,1100006,100003,2021-06-28,Hire,NaN,Active,"Hired into department_id=1, location_id=3, job..."
6,1100007,100004,2022-03-24,Hire,NaN,Active,"Hired into department_id=1, location_id=1, job..."
7,1100008,100005,2023-05-10,Hire,NaN,Active,"Hired into department_id=1, location_id=1, job..."
8,1100009,100006,2021-01-14,Hire,NaN,Active,"Hired into department_id=1, location_id=4, job..."
9,1100010,100007,2022-11-28,Hire,NaN,Active,"Hired into department_id=1, location_id=1, job..."


In [3]:
employee_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 16008 entries, 0 to 16007
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   event_id     16008 non-null  int64         
 1   employee_id  16008 non-null  int64         
 2   event_date   16008 non-null  datetime64[us]
 3   event_type   16008 non-null  str           
 4   old_value    6008 non-null   str           
 5   new_value    16008 non-null  str           
 6   notes        16008 non-null  str           
dtypes: datetime64[us](1), int64(2), str(4)
memory usage: 875.6 KB


## 2. Event-type counts

In [4]:
event_type_counts = (
    employee_events[
        "event_type"
    ]
    .value_counts()
    .rename_axis("event_type")
    .reset_index(name="event_count")
)

event_type_counts

,event_type,event_count
0,Hire,10000
1,Promotion,2146
2,Termination,1713
3,Leave,964
4,Manager Change,632
5,Transfer,553


## 3. Connect events to employees

In [5]:
employee_details = (
    employees[
        [
            "employee_id",
            "hire_date",
            "termination_date",
            "termination_type",
            "employment_status",
            "department_id",
            "location_id",
            "manager_id",
            "organizational_level",
        ]
    ]
    .copy()
)

employee_details[
    "employment_end_date"
] = (
    employee_details[
        "termination_date"
    ]
    .fillna(
        pd.Timestamp("2026-06-30")
    )
)

event_details = (
    employee_events
    .merge(
        employee_details,
        on="employee_id",
        how="left",
    )
)

event_details.head()

,event_id,employee_id,event_date,event_type,old_value,new_value,notes,hire_date,termination_date,termination_type,employment_status,department_id,location_id,manager_id,organizational_level,employment_end_date
0,1100001,100001,2022-10-17,Hire,NaN,Active,"Hired into department_id=1, location_id=1, job...",2022-10-17,NaT,NaN,Active,1,1,<NA>,Department Head,2026-06-30
1,1100002,100002,2022-04-03,Hire,NaN,Active,"Hired into department_id=1, location_id=1, job...",2022-04-03,NaT,NaN,Active,1,1,100001,Senior Manager,2026-06-30
2,1100003,100002,2024-04-15,Transfer,3,1,Internal location transfer; old_value and new_...,2022-04-03,NaT,NaN,Active,1,1,100001,Senior Manager,2026-06-30
3,1100004,100002,2025-01-22,Leave,Active,Leave,Medical Leave; planned duration 90 days.,2022-04-03,NaT,NaN,Active,1,1,100001,Senior Manager,2026-06-30
4,1100005,100002,2025-04-22,Leave,Leave,Active,Medical Leave; planned duration 90 days.,2022-04-03,NaT,NaN,Active,1,1,100001,Senior Manager,2026-06-30


## 4. Key and date checks

In [6]:
key_and_date_checks = pd.Series(
    {
        "table has seven columns": (
            len(
                employee_events.columns
            )
            == 7
        ),
        "event IDs are complete": (
            employee_events[
                "event_id"
            ].notna().all()
        ),
        "event IDs are unique": (
            employee_events[
                "event_id"
            ].is_unique
        ),
        "employee IDs are valid": (
            set(
                employee_events[
                    "employee_id"
                ]
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
        "employee-date-type combinations are unique": (
            not employee_events
            .duplicated(
                subset=[
                    "employee_id",
                    "event_date",
                    "event_type",
                ]
            )
            .any()
        ),
        "events do not occur before hire": (
            event_details[
                "event_date"
            ]
            .ge(
                event_details[
                    "hire_date"
                ]
            )
            .all()
        ),
        "events do not occur after employment": (
            event_details[
                "event_date"
            ]
            .le(
                event_details[
                    "employment_end_date"
                ]
            )
            .all()
        ),
    },
    name="passed",
)

key_and_date_checks

table has seven columns                       True
event IDs are complete                        True
event IDs are unique                          True
employee IDs are valid                        True
employee-date-type combinations are unique    True
events do not occur before hire               True
events do not occur after employment          True
Name: passed, dtype: bool

## 5. Hire-event checks

In [7]:
hire_events = employee_events[
    employee_events[
        "event_type"
    ]
    == "Hire"
]

hire_counts = (
    hire_events
    .groupby("employee_id")
    .size()
    .reindex(
        employees[
            "employee_id"
        ],
        fill_value=0,
    )
)

hire_details = (
    hire_events
    .merge(
        employees[
            [
                "employee_id",
                "hire_date",
            ]
        ],
        on="employee_id",
        how="left",
    )
)

hire_checks = pd.Series(
    {
        "there are 10,000 hire events": (
            len(hire_events)
            == 10_000
        ),
        "every employee has one hire event": (
            hire_counts.eq(1).all()
        ),
        "hire-event dates match hire dates": (
            hire_details[
                "event_date"
            ]
            .eq(
                hire_details[
                    "hire_date"
                ]
            )
            .all()
        ),
        "hire old values are blank": (
            hire_events[
                "old_value"
            ].isna().all()
        ),
        "hire new values are Active": (
            hire_events[
                "new_value"
            ].eq("Active").all()
        ),
    },
    name="passed",
)

hire_checks

there are 10,000 hire events         True
every employee has one hire event    True
hire-event dates match hire dates    True
hire old values are blank            True
hire new values are Active           True
Name: passed, dtype: bool

## 6. Termination-event checks

In [8]:
termination_events = employee_events[
    employee_events[
        "event_type"
    ]
    == "Termination"
]

terminated_employees = employees[
    employees[
        "employment_status"
    ]
    == "Terminated"
]

active_employees = employees[
    employees[
        "employment_status"
    ]
    == "Active"
]

termination_counts = (
    termination_events
    .groupby("employee_id")
    .size()
    .reindex(
        employees[
            "employee_id"
        ],
        fill_value=0,
    )
)

termination_details = (
    termination_events
    .merge(
        employees[
            [
                "employee_id",
                "termination_date",
            ]
        ],
        on="employee_id",
        how="left",
    )
)

termination_checks = pd.Series(
    {
        "termination count matches terminated employees": (
            len(termination_events)
            == len(terminated_employees)
        ),
        "terminated employees have one event": (
            termination_counts.loc[
                terminated_employees[
                    "employee_id"
                ]
            ].eq(1).all()
        ),
        "active employees have no termination event": (
            termination_counts.loc[
                active_employees[
                    "employee_id"
                ]
            ].eq(0).all()
        ),
        "termination dates match employee data": (
            termination_details[
                "event_date"
            ]
            .eq(
                termination_details[
                    "termination_date"
                ]
            )
            .all()
        ),
    },
    name="passed",
)

termination_checks

termination count matches terminated employees    True
terminated employees have one event               True
active employees have no termination event        True
termination dates match employee data             True
Name: passed, dtype: bool

## 7. Promotion-event checks

In [9]:
expected_promotions = (
    compensation_history[
        compensation_history[
            "change_reason"
        ]
        == "Promotion"
    ][
        [
            "employee_id",
            "effective_date",
        ]
    ]
    .copy()
)

promotion_events = employee_events[
    employee_events[
        "event_type"
    ]
    == "Promotion"
]

expected_promotion_pairs = set(
    expected_promotions[
        [
            "employee_id",
            "effective_date",
        ]
    ].itertuples(
        index=False,
        name=None,
    )
)

actual_promotion_pairs = set(
    promotion_events[
        [
            "employee_id",
            "event_date",
        ]
    ].itertuples(
        index=False,
        name=None,
    )
)

promotion_checks = pd.Series(
    {
        "promotion count matches compensation records": (
            len(promotion_events)
            == len(expected_promotions)
        ),
        "promotion employee-date pairs match": (
            actual_promotion_pairs
            == expected_promotion_pairs
        ),
        "promotion salaries do not decrease": (
            promotion_events[
                "new_value"
            ]
            .astype(int)
            .ge(
                promotion_events[
                    "old_value"
                ].astype(int)
            )
            .all()
        ),
    },
    name="passed",
)

promotion_checks

promotion count matches compensation records    True
promotion employee-date pairs match             True
promotion salaries do not decrease              True
Name: passed, dtype: bool

## 8. Transfer-event checks

In [10]:
transfer_events = (
    employee_events[
        employee_events[
            "event_type"
        ]
        == "Transfer"
    ]
    .merge(
        employees[
            [
                "employee_id",
                "location_id",
            ]
        ],
        on="employee_id",
        how="left",
    )
)

if len(transfer_events) > 0:
    transfer_old_locations = (
        transfer_events[
            "old_value"
        ].astype(int)
    )

    transfer_new_locations = (
        transfer_events[
            "new_value"
        ].astype(int)
    )

    transfer_checks = pd.Series(
        {
            "old and new locations differ": (
                transfer_old_locations
                .ne(
                    transfer_new_locations
                )
                .all()
            ),
            "new locations match current employee data": (
                transfer_new_locations
                .eq(
                    transfer_events[
                        "location_id"
                    ]
                )
                .all()
            ),
            "old location IDs are valid": (
                set(
                    transfer_old_locations
                )
                .issubset(
                    set(
                        locations[
                            "location_id"
                        ]
                    )
                )
            ),
            "new location IDs are valid": (
                set(
                    transfer_new_locations
                )
                .issubset(
                    set(
                        locations[
                            "location_id"
                        ]
                    )
                )
            ),
        },
        name="passed",
    )
else:
    transfer_checks = pd.Series(
        {
            "transfer records exist": False
        },
        name="passed",
    )

transfer_checks

old and new locations differ                 True
new locations match current employee data    True
old location IDs are valid                   True
new location IDs are valid                   True
Name: passed, dtype: bool

## 9. Manager-change checks

In [11]:
manager_change_events = (
    employee_events[
        employee_events[
            "event_type"
        ]
        == "Manager Change"
    ]
    .merge(
        employees[
            [
                "employee_id",
                "manager_id",
                "department_id",
            ]
        ],
        on="employee_id",
        how="left",
    )
)

manager_change_checks = pd.Series(
    {
        "manager-change records exist": (
            len(manager_change_events) > 0
        ),
        "old and new manager IDs differ": (
            manager_change_events[
                "old_value"
            ]
            .astype(int)
            .ne(
                manager_change_events[
                    "new_value"
                ].astype(int)
            )
            .all()
        ),
        "new manager matches current employee data": (
            manager_change_events[
                "new_value"
            ]
            .astype(int)
            .eq(
                manager_change_events[
                    "manager_id"
                ].astype(int)
            )
            .all()
        ),
        "old manager IDs are valid": (
            set(
                manager_change_events[
                    "old_value"
                ].astype(int)
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
        "new manager IDs are valid": (
            set(
                manager_change_events[
                    "new_value"
                ].astype(int)
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
    },
    name="passed",
)

manager_change_checks

manager-change records exist                 True
old and new manager IDs differ               True
new manager matches current employee data    True
old manager IDs are valid                    True
new manager IDs are valid                    True
Name: passed, dtype: bool

## 10. Leave-event checks

In [12]:
leave_events = employee_events[
    employee_events[
        "event_type"
    ]
    == "Leave"
]

leave_counts = (
    leave_events
    .groupby("employee_id")
    .size()
)

leave_start_events = leave_events[
    leave_events[
        "old_value"
    ].eq("Active")
    & leave_events[
        "new_value"
    ].eq("Leave")
]

leave_return_events = leave_events[
    leave_events[
        "old_value"
    ].eq("Leave")
    & leave_events[
        "new_value"
    ].eq("Active")
]

leave_periods = (
    leave_start_events[
        [
            "employee_id",
            "event_date",
        ]
    ]
    .merge(
        leave_return_events[
            [
                "employee_id",
                "event_date",
            ]
        ],
        on="employee_id",
        suffixes=(
            "_start",
            "_return",
        ),
    )
)

leave_checks = pd.Series(
    {
        "leave-event total is even": (
            len(leave_events) % 2 == 0
        ),
        "every leave employee has two events": (
            leave_counts.eq(2).all()
        ),
        "every leave has one start": (
            leave_start_events[
                "employee_id"
            ].nunique()
            == leave_counts.size
        ),
        "every leave has one return": (
            leave_return_events[
                "employee_id"
            ].nunique()
            == leave_counts.size
        ),
        "returns occur after leave starts": (
            leave_periods[
                "event_date_return"
            ]
            .gt(
                leave_periods[
                    "event_date_start"
                ]
            )
            .all()
        ),
    },
    name="passed",
)

leave_checks

leave-event total is even              True
every leave employee has two events    True
every leave has one start              True
every leave has one return             True
returns occur after leave starts       True
Name: passed, dtype: bool

## 11. Event summary

In [13]:
employee_events[
    "event_year"
] = (
    employee_events[
        "event_date"
    ].dt.year
)

events_by_year = (
    employee_events
    .groupby(
        [
            "event_year",
            "event_type",
        ]
    )
    .size()
    .rename("event_count")
    .reset_index()
)

events_by_year

,event_year,event_type,event_count
0,2021,Hire,1924
1,2021,Leave,5
2,2021,Termination,16
3,2021,Transfer,2
4,2022,Hire,1839
5,2022,Leave,71
6,2022,Manager Change,4
7,2022,Promotion,182
8,2022,Termination,104
9,2022,Transfer,32


In [14]:
events_per_employee = (
    employee_events
    .groupby("employee_id")
    .size()
    .sort_values(
        ascending=False
    )
    .rename("event_count")
)

events_per_employee.head(10)

employee_id
104049    7
103972    6
104285    6
106375    6
103813    6
102871    6
103812    6
103798    6
109358    6
104896    5
Name: event_count, dtype: int64

In [15]:
employee_with_most_events = (
    events_per_employee.index[0]
)

employee_events[
    employee_events[
        "employee_id"
    ]
    == employee_with_most_events
].sort_values(
    "event_date"
)

,event_id,employee_id,event_date,event_type,old_value,new_value,notes,event_year
6491,1106492,104049,2021-11-13,Hire,NaN,Active,"Hired into department_id=2, location_id=1, job...",2021
6492,1106493,104049,2022-08-19,Transfer,5,1,Internal location transfer; old_value and new_...,2022
6493,1106494,104049,2022-11-13,Promotion,105400,118100,Promotion-related compensation change; old_val...,2022
6494,1106495,104049,2024-02-29,Leave,Active,Leave,Medical Leave; planned duration 84 days.,2024
6495,1106496,104049,2024-05-23,Leave,Leave,Active,Medical Leave; planned duration 84 days.,2024
6496,1106497,104049,2024-11-13,Promotion,123800,135000,Promotion-related compensation change; old_val...,2024
6497,1106498,104049,2026-04-14,Termination,Active,Terminated,Voluntary termination.,2026


## 12. Complete validation summary

In [16]:
basic_checks = pd.Series(
    {
        "all 10,000 employees have events": (
            employee_events[
                "employee_id"
            ].nunique()
            == 10_000
        ),
        "event types are valid": (
            set(
                employee_events[
                    "event_type"
                ]
            )
            .issubset(
                {
                    "Hire",
                    "Promotion",
                    "Transfer",
                    "Manager Change",
                    "Leave",
                    "Termination",
                }
            )
        ),
    },
    name="passed",
)

all_checks = pd.concat(
    [
        basic_checks,
        key_and_date_checks,
        hire_checks,
        termination_checks,
        promotion_checks,
        transfer_checks,
        manager_change_checks,
        leave_checks,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,"all 10,000 employees have events",True
1,event types are valid,True
2,table has seven columns,True
3,event IDs are complete,True
4,event IDs are unique,True
5,employee IDs are valid,True
6,employee-date-type combinations are unique,True
7,events do not occur before hire,True
8,events do not occur after employment,True
9,"there are 10,000 hire events",True


In [17]:
if validation_results["passed"].all():
    print(
        "All employee-event validation "
        "checks passed."
    )
else:
    print(
        "One or more employee-event "
        "validation checks failed."
    )

All employee-event validation checks passed.


## 13. Conclusions

The synthetic employee-event table successfully represents major workforce events.

### Successful checks

- Event IDs are complete and unique.
- Employee foreign keys are valid.
- Events remain inside employee employment periods.
- Every employee has exactly one hire event.
- Hire-event dates match employee hire dates.
- Terminated employees have exactly one termination event.
- Active employees have no termination events.
- Promotion events agree with compensation history.
- Transfer events use valid and different location IDs.
- Manager changes use valid manager IDs.
- Leave periods contain both a start and return event.
- All employee-event validation checks passed.

### Current simplifications

- Transfers represent location changes rather than department changes.
- At most one transfer is generated for each selected employee.
- At most one manager change is generated for each selected employee.
- At most one leave period is generated for each selected employee.
- Historical promotion events use salary changes because historical job-role records do not yet exist.
- The current employee table stores only the employee's latest manager, department, location, and job role.